# SE201 — LangChain and LangGraph: Companion Notebook

This notebook accompanies the LangChain and LangGraph lecture. Each section is a small, self-contained example of a pattern you are likely to need in your group project, built on an **open-weight model served by Ollama** — on your own computer, or on a server reached through an SSH tunnel (Section 0). No commercial API keys are used anywhere.

| Section | Pattern | Typical project use |
|---|---|---|
| 1 | Calling a model, prompt templates, chains | Any feature that sends a prompt and shows the reply |
| 2 | Structured output into a Pydantic model | Turning free text (issues, logs, reviews) into data your code can act on |
| 3 | Retrieval-augmented generation (RAG) | Answering questions over your own documentation or codebase |
| 4 | A tool-using agent with `create_agent` | Letting the model inspect files and run tests to answer a question |
| 5 | Middleware: human approval and call limits | Putting a person in charge of any action that changes something |
| 6 | A custom LangGraph graph | A generate → test → repair loop with a retry limit |
| 7 | Conversation memory with a checkpointer | A chat feature that remembers earlier turns per user |
| 8 | Choosing the right level of abstraction | Deciding what your project actually needs |

**Relationship to Week 6.** In the Week 6 notebook you wrote the tool-calling loop and the retrieval pipeline by hand. Everything here does the same jobs; the point is to see which parts the frameworks take over and which parts remain your responsibility.

**Versions.** Written against `langchain` 1.x and `langgraph` 1.x. Much of the material you will find online uses the pre-1.0 API (`AgentExecutor`, `LLMChain`, `initialize_agent`, `langgraph.prebuilt.create_react_agent`). Those are legacy; do not copy them into your project.

## 0. Prerequisites

In a terminal, on whichever machine runs the daemon (see *Where Ollama runs* below):

```bash
ollama pull qwen2.5:7b          # chat model; must support tool calling
ollama pull nomic-embed-text    # embedding model for Section 3 (the same one as Week 6)
ollama list                     # confirm both landed
```

The health-check cell pulls anything still missing, but doing it in advance avoids a long wait in class.

**Choosing the chat model.** Sections 2, 4 and 5 rely on *tool calling*, which not every local model supports. `qwen2.5:7b` handles it reliably on a laptop with 16 GB of RAM. Smaller models (for example `llama3.2:3b`) will run, but expect them to call the wrong tool or skip tools entirely — which is itself a useful thing to observe. If you change `MODEL`, re-run every section and compare.

Python packages: `langchain`, `langgraph`, `langchain-ollama` (LangChain's wrapper around the `ollama` client you used in Weeks 5 and 6) and `langchain-text-splitters`.

In [ ]:
# Run once. --quiet keeps the notebook output tidy.
%pip install --quiet --upgrade "langchain>=1.0" "langgraph>=1.0" langchain-ollama langchain-text-splitters

### Where Ollama runs

The same two options as Week 6, and the same variable. Every LangChain object in this notebook talks to `HOST`.

**(a) Locally.** The daemon on this machine, `http://127.0.0.1:11434`. Leave `HOST` alone.

**(b) On another machine, through an SSH tunnel.** Useful when this laptop cannot run the models, or a shared machine with a GPU is available. In a terminal, open the tunnel and leave it running:

```bash
ssh -N -L 11435:127.0.0.1:11434 you@gpu-box      # add -f to background it
```

This forwards local port 11435 to the daemon's loopback port on `gpu-box`. The remote daemon keeps listening on its own `127.0.0.1` only; nothing is exposed to the network, and no `OLLAMA_HOST=0.0.0.0` is needed on the remote side. Then set `HOST` to the local end of the tunnel, `http://127.0.0.1:11435`. Using 11435 rather than 11434 leaves room for a local daemon on the same laptop. `ollama pull` above must be run *on the remote machine* (or use the health-check cell below, which asks the daemon at `HOST` to pull).

The tunnel lasts as long as that SSH connection. If the laptop sleeps or the network changes, cells start failing with connection errors: re-run the `ssh` command, then the health check.

In [ ]:
import os
import ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings

# (a) local daemon: leave as is.   (b) SSH tunnel: set HOST to the local end of the tunnel.
HOST = os.environ.get("OLLAMA_HOST") or "http://127.0.0.1:11434"
#HOST = "http://127.0.0.1:11435"          # <- (b): uncomment after opening the tunnel

os.environ["OLLAMA_HOST"] = HOST           # any Ollama client created later without base_url uses the same daemon
MODEL = "qwen2.5:7b"
#MODEL = "qwen3:8b"                         # generation; must support tool calling
EMBED = "nomic-embed-text"                 # embeddings

client = ollama.Client(host=HOST)          # used only for the health check below

In [ ]:
# Health check: is a daemon answering at HOST, and are both models there? Pull any that are missing.
# (A pull is a no-op if the model is present. With a tunnel, the pull happens on the remote machine.)
try:
    names = [m.model for m in client.list().models]
except Exception as e:
    raise SystemExit(f"Nothing answering at {HOST}.\n"
                     f"  local:  start the daemon with `ollama serve`\n"
                     f"  tunnel: check `ssh -N -L 11435:127.0.0.1:11434 you@gpu-box` is running and HOST matches\n"
                     f"  ({e})")

print(f"daemon at {HOST}\ninstalled models:")
for n in names:
    print("  ", n)

def ensure_model(name):
    if name in names or f"{name}:latest" in names:       # exact tag: qwen2.5:14b does not count as qwen2.5:7b
        return
    print(f"\n{name} is not pulled — pulling now (a few GB; be patient)")
    last_pct, last_status = -100, None
    for p in client.pull(name, stream=True):        # streams progress objects: status, total, completed
        if p.total and p.completed:
            pct = 100 * p.completed / p.total
            if pct - last_pct >= 10 or pct == 100:
                print(f"  {p.status:<28} {pct:5.1f}%"); last_pct = pct
        elif p.status != last_status:
            print(f"  {p.status}"); last_status = p.status

for want in (MODEL, EMBED):
    ensure_model(want)

### The LangChain model object, and what it shows you

From here on, the notebook does not call `client` directly. It uses LangChain's `ChatOllama`, created once, pointed at the same `HOST`.

Two switches in the next cell control what you see on every call:

* **`SHOW_REQUESTS`** (on by default). Every time LangChain calls the model, a compact copy of the request is printed *before* the reply: the settings, the tools offered, the output format, and every message sent. This is what the model actually receives, which is often more than the code you wrote suggests. Set it to `False` to turn the printing off; it takes effect on the next call, with no need to re-run anything. The requests are still recorded in `requests_sent` either way.
* **`THINK`** (off by default). qwen3 can reason before it answers. With `THINK = True` the reasoning is kept, and `show()` prints it between `<think>` tags before the answer, as `answer_of()` did in Week 6. Thinking makes every call slower, and the agent sections make several calls per run. After changing `THINK`, re-run this cell, the next one, and every cell that builds a chain or agent from `llm`.

`show(reply)` is used wherever the notebook prints a model reply. Chains that end in `StrOutputParser` return only the answer text, so `show()` has no reasoning to print for them.

In [ ]:
import json, textwrap
from collections import deque

SHOW_REQUESTS = True    # False: stop printing each request sent to the model (still recorded in requests_sent)
THINK = False           # True: the model reasons before answering (slower); show() prints the reasoning

requests_sent = deque(maxlen=100)     # every request body sent to the chat model, most recent last

def wrap(text, width=100):
    """Print long text wrapped to width, keeping paragraph breaks."""
    for para in str(text).split("\n"):
        print(textwrap.fill(para, width) if para.strip() else "")

def show(reply, label=None):
    """Print a model reply: the reasoning first (if THINK kept any), then the answer.
    Accepts an AIMessage or a plain string (the output of a chain ending in StrOutputParser)."""
    if label:
        print(f"{label}:")
    if isinstance(reply, str):
        wrap(reply)
        return
    reasoning = reply.additional_kwargs.get("reasoning_content", "")
    if reasoning.strip():
        print("<think>"); wrap(reasoning.strip()); print("</think>\n")
    wrap(reply.content)

def _one_line(text, width=110):
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[:width - 1] + "…"

def print_request(body):
    """Compact view of one request to Ollama's /api/chat endpoint."""
    settings = ", ".join([f"think={body.get('think')}"] + [f"{k}={v}" for k, v in body.get("options", {}).items()])
    print(f"┌─ request to {body['model']} ({settings})")
    if body.get("tools"):
        print("│  tools offered: " + ", ".join(t["function"]["name"] for t in body["tools"]))
    if isinstance(body.get("format"), dict):
        print("│  output format: JSON schema with fields " + ", ".join(body["format"].get("properties", {})))
    for m in body["messages"]:
        if m.get("tool_calls"):
            calls = "; ".join(f"{c['function']['name']}({c['function'].get('arguments', {})})" for c in m["tool_calls"])
            print(f"│  {m['role']:<9} → {_one_line(calls)}")
        else:
            print(f"│  {m['role']:<9} {_one_line(m.get('content', ''))}")
    print("└─")

def record_request(request):
    """httpx event hook: runs just before each HTTP request leaves for Ollama."""
    if request.url.path == "/api/chat":
        body = json.loads(request.content)
        requests_sent.append(body)
        if SHOW_REQUESTS:
            print_request(body)

In [ ]:
# num_ctx raises Ollama's context window; agents that read files need the room.
# client_kwargs attaches record_request to the HTTP client, so every request passes through it.

if MODEL.startswith("qwen2.5"):
    # No thinking mode: send no think flag, decode greedily.
    THINK = False
    reasoning, sampling = None, dict(temperature=0)
elif THINK:
    # qwen3 thinking: Qwen's recommended sampling (greedy decoding loops in thinking mode).
    # num_predict caps the reply, reasoning included, so a loop cannot run forever.
    reasoning, sampling = True, dict(temperature=0.6, top_p=0.95, top_k=20, num_predict=6144)
else:
    # qwen3 without thinking: a low temperature, so runs are close to repeatable.
    reasoning, sampling = False, dict(temperature=0.3)

llm = ChatOllama(model=MODEL, base_url=HOST, num_ctx=8192, reasoning=reasoning,
                 client_kwargs={"event_hooks": {"request": [record_request]}},
                 **sampling)

show(llm.invoke("Reply with the single word OK."))

## 1. Calling a local model through LangChain

**What this replaces.** In Week 6 you called the `ollama` client directly and built the message list yourself. LangChain wraps every provider in the same chat-model interface, so the calling code does not change when the model or provider changes.

**Three things to notice:**
1. Messages are typed objects (`SystemMessage`, `HumanMessage`, `AIMessage`) rather than dictionaries.
2. The reply is an `AIMessage`; its text is in `.content`, and it also carries metadata such as token counts.
3. Every chat model supports `.invoke`, `.stream` and `.batch` — the same methods you will see on chains and agents later.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

reply = llm.invoke([
    SystemMessage("You are a concise senior software engineer."),
    HumanMessage("In two sentences, what is the difference between a unit test and an integration test?"),
])
show(reply)
print("\nToken usage:", reply.usage_metadata)

If `THINK = True`, `show()` prints the reasoning above the answer. LangChain keeps it in `reply.additional_kwargs["reasoning_content"]`, separate from `reply.content`, so code that uses the answer never has to strip `<think>` tags out of it.

**Streaming.** For anything user-facing, stream the reply so the first words appear immediately rather than after the whole answer is generated.

In [ ]:
# Streams the answer only: with THINK = True, reasoning chunks arrive first with empty content.
for chunk in llm.stream("List three common causes of flaky tests, one line each."):
    print(chunk.content, end="", flush=True)
print()

### 1.1 Prompt templates and chains

A **prompt template** separates the fixed wording of a prompt from the values that change on each call. Joining a template, a model and an output parser with `|` produces a **chain**: a single object you can invoke, stream or batch like a model.

The example below is a reusable "explain this error to a junior developer" feature — the kind of thing a developer-tools project might expose behind a button.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain error messages to junior developers. "
               "Give the likely cause and one concrete fix. Maximum 80 words. "
               "The programming language is {language}."),
    ("human", "{error}"),
])

explain_error = explain_prompt | llm | StrOutputParser()

show(explain_error.invoke({
    "language": "Python",
    "error": "TypeError: 'NoneType' object is not subscriptable",
}))

`.batch` runs several inputs, in parallel where the backend allows it. With a local model the speed-up is limited by your hardware, but the code is the same as against a hosted service.

In [ ]:
errors = [
    {"language": "Python", "error": "ModuleNotFoundError: No module named 'requests'"},
    {"language": "JavaScript", "error": "TypeError: Cannot read properties of undefined (reading 'map')"},
]
for item, answer in zip(errors, explain_error.batch(errors)):
    print(f"--- {item['error']}")
    show(answer)
    print()

## 2. Structured output: turning free text into data

**Problem.** Parsing free text with regular expressions is brittle.

**Approach.** Describe the shape you want as a Pydantic model and call `with_structured_output`. LangChain passes the schema to the model and validates the reply into an instance of your class. The field descriptions are sent to the model, so write them as instructions.

The example triages incoming issue reports — a common building block in SE-tooling projects (issue bots, support dashboards, release-note generators).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class IssueTriage(BaseModel):
    '''Triage decision for a single issue report.'''
    category: Literal["bug", "feature_request", "question", "documentation"]
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="critical = data loss or security; high = core feature broken; "
                    "medium = workaround exists; low = cosmetic")
    component: str = Field(description="Part of the system affected, e.g. 'login', 'checkout', 'build'")
    summary: str = Field(description="One-sentence neutral summary of the report")
    needs_more_info: bool = Field(description="True if the report lacks steps to reproduce or expected behaviour")

triage_prompt = ChatPromptTemplate.from_messages([
    ("system", "You triage issue reports for a small e-commerce web application."),
    ("human", "Issue report:\n{report}"),
])

triage = triage_prompt | llm.with_structured_output(IssueTriage)

In [ ]:
reports = [
    "Checkout page crashes with a 500 error whenever the cart has more than 20 items. "
    "Steps: add 21 items, press Pay. Expected: payment page. Happens every time.",
    "It would be nice if we could export order history as CSV.",
    "login broken??",
    "The README still says to run `npm start` but the script was renamed to `npm run dev`.",
]

results = []
for report in reports:
    try:
        results.append(triage.invoke({"report": report}))
    except Exception as exc:   # small models sometimes return invalid JSON; never assume success
        print(f"Could not triage: {report[:40]!r} ({type(exc).__name__})")

for r in results:
    print(f"{r.category:16} {r.severity:9} {r.component:12} more info: {str(r.needs_more_info):5} | {r.summary}")

Because each result is a Pydantic object, the rest of your program is ordinary Python:

In [ ]:
urgent = [r for r in results if r.severity in ("high", "critical")]
print(f"{len(urgent)} urgent issue(s)")
print(results[0].model_dump_json(indent=2) if results else "no results")

**The schema the model was given.** The request printouts above list the fields of an output format. This is the whole of it, as sent with the last triage call: the JSON schema Pydantic generated from `IssueTriage`, including every `Field` description. Ollama uses it to restrict the model's output to JSON of this shape.

In [ ]:
print(json.dumps(requests_sent[-1]["format"], indent=2))

**What the framework does not do for you.** Validation guarantees the *shape*, not the *judgement*. A schema-valid reply can still put a security bug at "low". If your project relies on a classification, build a small labelled test set and measure agreement — the evaluation ideas from earlier weeks apply directly.

## 3. Retrieval-augmented generation over your own documents

This is the Week 6 pipeline expressed with LangChain components. Each step maps one-to-one onto code you wrote by hand:

| Week 6 (by hand) | LangChain component |
|---|---|
| Read files into strings | `Document` objects (or a document loader) |
| Split into chunks | `RecursiveCharacterTextSplitter` |
| Call the embedding model | `OllamaEmbeddings` |
| Store vectors, cosine similarity search | `InMemoryVectorStore` → `.as_retriever()` |
| Build the prompt with retrieved text | A prompt template fed by the retriever |

The corpus below is a fictional project's documentation, kept inline so the notebook runs anywhere. For your project, replace it with your own files (see the cell after the example).

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(metadata={"source": "README.md"}, page_content='''
ShopLite is a small e-commerce web application written in Python with FastAPI and a PostgreSQL database.
To run it locally, copy .env.example to .env, start the database with `docker compose up db`, then run
`uvicorn shoplite.main:app --reload`. The application listens on port 8000. Seed data can be loaded with
`python -m shoplite.seed`, which creates twenty products and two test users.'''),
    Document(metadata={"source": "CONTRIBUTING.md"}, page_content='''
Branch names follow the pattern type/short-description, for example fix/cart-total or feat/csv-export.
Every pull request needs at least one approving review and a green CI run. New code must include unit tests;
the CI job fails if line coverage drops below 80 per cent. Commit messages use the imperative mood.
Do not commit secrets: the pre-commit hook runs gitleaks and will block the commit.'''),
    Document(metadata={"source": "docs/api.md"}, page_content='''
The REST API is versioned under /api/v1. GET /api/v1/products returns a paginated list; use the page and
size query parameters, with a maximum size of 100. POST /api/v1/cart/items adds an item and requires a
bearer token. Orders are created with POST /api/v1/orders; the endpoint is idempotent if the client sends
an Idempotency-Key header. Errors use RFC 9457 problem details.'''),
    Document(metadata={"source": "docs/deploy.md"}, page_content='''
Production deploys happen automatically when a tag matching v*.*.* is pushed to main. The pipeline builds a
container image, runs database migrations with Alembic, and performs a rolling update. To roll back,
re-deploy the previous tag; migrations must therefore be backwards compatible for one release.
Staging is redeployed on every merge to main.'''),
]
print(len(docs), "documents")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50,
    separators=["\n\n", ". ", "\n", " ", ""], keep_separator="end",
)
chunks = splitter.split_documents(docs)          # metadata (the source file) is copied onto every chunk
print(len(chunks), "chunks")

embeddings = OllamaEmbeddings(model=EMBED, base_url=HOST)
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

**Check retrieval before generation.** If the right chunk is not retrieved, no prompt will rescue the answer. Always inspect what the retriever returns for a few representative questions.

In [ ]:
for doc in retriever.invoke("How do I roll back a bad release?"):
    print(f"[{doc.metadata['source']}] {doc.page_content[:90]}...")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(retrieved):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in retrieved)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. Cite the source file in square brackets after each claim. "
               "If the context does not contain the answer, say you do not know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

for q in ["How do I roll back a bad release?",
          "What is the maximum page size for the products endpoint?",
          "Which payment provider does ShopLite use?"]:          # not in the documents: should say it does not know
    print(f"Q: {q}")
    show(rag_chain.invoke(q))
    print()

## 4. A tool-using agent with `create_agent`

**What this replaces.** The Week 6 loop: send messages with tool schemas → the model asks for a tool → you run it → append the result → repeat until the model answers in text. `create_agent` is that loop, running on the LangGraph runtime.

**The scenario.** A tiny repository with a failing test. The agent gets three *read-only* tools and is asked to diagnose the failure. This is the core of many project ideas: code-review assistants, test-failure explainers, onboarding bots.

First, create the sample repository. It contains a deliberate bug.

In [ ]:
import subprocess, sys, tempfile, textwrap
from pathlib import Path
REPO = Path(tempfile.mkdtemp(prefix="se201_repo_"))

(REPO / "pricing.py").write_text(textwrap.dedent('''
    def apply_discount(price: float, percent: float) -> float:
        # Return the price after a percentage discount, rounded to 2 decimal places.
        if not 0 <= percent <= 100:
            raise ValueError("percent must be between 0 and 100")
        return round(price - price * percent, 2)
''').lstrip())

(REPO / "test_pricing.py").write_text(textwrap.dedent('''
    import unittest
    from pricing import apply_discount

    class TestDiscount(unittest.TestCase):
        def test_ten_percent(self):
            self.assertEqual(apply_discount(200.0, 10), 180.0)

        def test_zero(self):
            self.assertEqual(apply_discount(99.99, 0), 99.99)

        def test_rejects_negative(self):
            with self.assertRaises(ValueError):
                apply_discount(10.0, -5)

    if __name__ == "__main__":
        unittest.main()
''').lstrip())

print("Sample repository:", REPO)

**Tools are ordinary Python functions.** The `@tool` decorator turns the function name, type hints and docstring into the schema the model sees — so the docstring *is* the instruction to the model.

Note the `_safe_path` helper. A model choosing file paths is untrusted input: without the check, `read_file("../../.ssh/id_rsa")` would work. Any project tool that touches the file system, a shell or a database needs the same kind of guard.

In [ ]:
from langchain.tools import tool

def _safe_path(relative: str) -> Path:
    '''Resolve a path and refuse anything outside the sample repository.'''
    path = (REPO / relative).resolve()
    if not path.is_relative_to(REPO.resolve()):
        raise ValueError(f"Access outside the repository is not allowed: {relative}")
    return path

@tool
def list_files() -> str:
    '''List the files in the repository.'''
    return "\n".join(sorted(p.name for p in REPO.iterdir() if p.is_file()))

@tool
def read_file(path: str) -> str:
    '''Return the contents of a file in the repository. `path` is relative to the repository root.'''
    return _safe_path(path).read_text()

@tool
def run_tests() -> str:
    '''Run the repository's unit tests and return the test runner output.'''
    proc = subprocess.run([sys.executable, "-m", "unittest", "-v"], cwd=REPO,
                          capture_output=True, text=True, timeout=60)
    return (proc.stdout + proc.stderr)[-3000:]      # truncate: tool output consumes context window

print(run_tests.invoke({}))        # tools can be called directly, which is how you should unit-test them

In [ ]:
from langchain.agents import create_agent

diagnoser = create_agent(
    model=llm,
    tools=[list_files, read_file, run_tests],
    system_prompt=("You are a debugging assistant. Use the tools to investigate before answering. "
                   "Do not guess file contents. Explain the root cause and the minimal fix."),
)

result = diagnoser.invoke({"messages": [HumanMessage("The tests are failing. Find out why.")]})
show(result["messages"][-1])

The model receives each tool as a JSON schema, generated from the function name, the type hints and the docstring. This is `read_file` as the model received it; the docstring is sent word for word, so it is part of the prompt:

In [ ]:
print(json.dumps(next(t for t in requests_sent[-1]["tools"] if t["function"]["name"] == "read_file"), indent=2))

Even this is not the exact text the model reads. Ollama renders the messages and tool schemas through the model's *chat template* into a single sequence of tokens, with the model's role markers and the tool definitions written into the system turn. To see the template, run `ollama show qwen3:8b --template` in a terminal on the machine that runs the daemon.

**Inspect the trace.** The returned state holds every message, including each tool call and tool result. Reading it is how you find out *why* an agent gave a bad answer — and it is also a record of what the AI actually did, which is relevant to your project's disclosure artefact.

In [ ]:
def show_trace(messages):
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            for tc in m.tool_calls:
                print(f"AI    -> calls {tc['name']}({tc['args']})")
        elif m.type == "tool":
            print(f"TOOL  <- {m.name}: {m.content[:700]!r}...")
        else:
            print(f"{m.type.upper():5} {m.content[:1000]!r}")

show_trace(result["messages"])

This is how you log model interaction for development and auditing.

In [ ]:
import datetime

def save_trace(messages, path):
    '''Write an agent run to JSON so it can be reviewed or attached to project documentation.'''
    record = {"saved_at": datetime.datetime.now().isoformat(timespec="seconds"), "model": MODEL,
              "messages": [m.model_dump(include={"type", "content", "tool_calls", "name"}) for m in messages]}
    Path(path).write_text(json.dumps(record, indent=2, default=str))

save_trace(result["messages"], "trace_diagnose.json")
print("Saved trace_diagnose.json")

## 5. Middleware: human approval and call limits

A system prompt is a request, not a control. **Middleware** runs code at fixed points in the agent loop, so the behaviour is enforced by your program rather than by the model's cooperation.

This section gives the agent a `write_file` tool and adds two middleware components:

* `HumanInTheLoopMiddleware` pauses the run whenever the model calls `write_file`, so a person can approve, edit or reject the change before it happens.
* `ToolCallLimitMiddleware` caps the number of tool calls per run, which stops a confused model looping indefinitely.

Pausing requires a **checkpointer**: the agent's state is saved at the interrupt and the run resumes from that point once a decision arrives. The `thread_id` identifies which saved run to resume.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware, ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def write_file(path: str, content: str) -> str:
    '''Overwrite a file in the repository with new content. `path` is relative to the repository root.'''
    _safe_path(path).write_text(content)
    return f"Wrote {len(content)} characters to {path}"

fixer = create_agent(
    model=llm,
    tools=[list_files, read_file, run_tests, write_file],
    system_prompt=("You fix bugs. Read the relevant files, make the smallest change that makes the tests pass, "
                   "write the corrected file with write_file, then run the tests to confirm."),
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"write_file": True}),   # read-only tools are not gated
        ToolCallLimitMiddleware(run_limit=10),
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "fix-discount-1"}}
state = fixer.invoke({"messages": [HumanMessage("Fix the failing test in this repository.")]}, config)

If the model has proposed a write, the run is now paused. The interrupt carries the proposed tool calls for review:

In [ ]:
def pending_actions(state):
    interrupts = state.get("__interrupt__", [])
    return interrupts[0].value["action_requests"] if interrupts else []

for action in pending_actions(state):
    print(f"Proposed: {action['name']}  path={action['args'].get('path')}")
    print(action["args"].get("content", ""))

if not pending_actions(state):
    print("No write proposed — the agent finished without asking:")
    show(state["messages"][-1])

**You decide.** Set `DECISION` to `"approve"` to let the write happen, or `"reject"` to refuse it with a reason the model will see. Then resume. (In a real application this decision would come from a button in your interface.)

In [ ]:
DECISION = "approve"        # or "reject"

while pending_actions(state):
    decisions = []
    for _ in pending_actions(state):
        if DECISION == "approve":
            decisions.append({"type": "approve"})
        else:
            decisions.append({"type": "reject", "message": "Do not change the code; explain the fix instead."})
    state = fixer.invoke(Command(resume={"decisions": decisions}), config)

show(state["messages"][-1])

In [ ]:
# Do not trust the agent's own report of success: verify independently.
print(run_tests.invoke({}))

Other built-in middleware worth knowing for projects: `ModelCallLimitMiddleware` (caps model calls), `SummarizationMiddleware` (condenses long conversations to fit the context window), `PIIMiddleware` (redacts or blocks personal data in inputs and outputs), and `ModelFallbackMiddleware` (switches to a second model when the first fails). You can also write your own with the `@before_model`, `@after_model` and `@wrap_tool_call` decorators — for example, to log every tool call for your disclosure artefact.

## 6. A custom LangGraph graph: generate → test → repair

`create_agent` is one fixed shape: model, tools, repeat. When your workflow has a shape of its own — fixed steps, branches, a retry limit — you build the graph directly with LangGraph.

**The workflow.** Given a function specification and unit tests:

1. **generate** — the model writes the function (on a retry, it also sees the failing test output);
2. **test** — the code runs against the tests in a separate process;
3. **route** — if the tests pass, or the attempt limit is reached, stop; otherwise go back to *generate*.

The model never decides whether the code is correct; the tests do. That is the main reason to prefer an explicit graph here over a free-running agent.

A graph has three parts:
* **State** — a typed dictionary that every node reads and updates;
* **Nodes** — plain functions that take the state and return the fields they change;
* **Edges** — fixed (`add_edge`) or conditional (`add_conditional_edges`, which calls a routing function).

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class CodeState(TypedDict):
    spec: str
    tests: str
    code: str
    feedback: str     # test output from the previous attempt; empty on the first attempt
    attempts: int
    passed: bool

MAX_ATTEMPTS = 5

In [ ]:
import re

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write Python. Reply with a single ```python code block containing only the requested "
               "function and any imports it needs. No tests, no explanation."),
    ("human", "Specification:\n{spec}\n\nThe code must pass these tests:\n{tests}\n\n{feedback}"),
])

# Code generation uses a coding model; the generate node calls no tools, so tool calling does not matter.
CODE_MODEL = "qwen2.5-coder:7b"
gen_llm = ChatOllama(model=CODE_MODEL, base_url=HOST, num_ctx=8192, temperature=0.3,
                     client_kwargs={"event_hooks": {"request": [record_request]}})
gen_chain = gen_prompt | gen_llm | StrOutputParser()

def extract_code(text: str) -> str:
    '''Take the first fenced code block if there is one; models do not always follow format instructions.'''
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (match.group(1) if match else text).strip()

raw_replies = []                    # every raw model reply, for inspection

def generate(state: CodeState) -> dict:
    feedback = ""
    if state["feedback"]:
        feedback = ("Your previous attempt failed. Previous code:\n" + state["code"]
                    + "\n\nTest output:\n" + state["feedback"]
                    + "\nThat code is wrong. Do not repeat it: find the cause in the test output "
                      "and write a different implementation.")
    reply = gen_chain.invoke({"spec": state["spec"], "tests": state["tests"], "feedback": feedback})
    raw_replies.append(reply)
    return {"code": extract_code(reply), "attempts": state["attempts"] + 1}

def run_candidate(state: CodeState) -> dict:
    '''Run the generated code against the tests in a fresh process with a timeout.
    Generated code is untrusted: never exec() it inside the notebook process.'''
    with tempfile.TemporaryDirectory() as tmp:
        Path(tmp, "solution.py").write_text(state["code"])
        Path(tmp, "test_solution.py").write_text("from solution import *\n" + state["tests"])
        try:
            proc = subprocess.run([sys.executable, "-m", "unittest", "-q", "test_solution"], cwd=tmp,
                                  capture_output=True, text=True, timeout=20)
            output, passed = (proc.stdout + proc.stderr)[-2000:], proc.returncode == 0
        except subprocess.TimeoutExpired:
            output, passed = "Timed out after 20 seconds (possible infinite loop).", False
    return {"passed": passed, "feedback": "" if passed else output}

def route_after_test(state: CodeState) -> str:
    if state["passed"] or state["attempts"] >= MAX_ATTEMPTS:
        return "done"
    return "retry"

In [ ]:
builder = StateGraph(CodeState)
builder.add_node("generate", generate)
builder.add_node("test", run_candidate)
builder.add_edge(START, "generate")
builder.add_edge("generate", "test")
builder.add_conditional_edges("test", route_after_test, {"retry": "generate", "done": END})
code_graph = builder.compile()

# Mermaid source for the graph; paste into https://mermaid.live to view it as a diagram.
print(code_graph.get_graph().draw_mermaid())

In [ ]:
SPEC = '''def parse_duration(text: str) -> int
Convert a duration such as "1h30m", "45s", "2h" or "1h5m10s" into a whole number of seconds.
Units are h, m and s, always in that order, each optional but at least one present.
Each unit is an optional group: a number immediately followed by its letter.
Raise ValueError for anything else, including an empty string.'''

TESTS = '''
import unittest

class TestParseDuration(unittest.TestCase):
    def test_hours_minutes(self):   self.assertEqual(parse_duration("1h30m"), 5400)
    def test_seconds(self):         self.assertEqual(parse_duration("45s"), 45)
    def test_all_units(self):       self.assertEqual(parse_duration("1h5m10s"), 3910)
    def test_hours_only(self):      self.assertEqual(parse_duration("2h"), 7200)
    def test_empty(self):
        with self.assertRaises(ValueError): parse_duration("")
    def test_wrong_order(self):
        with self.assertRaises(ValueError): parse_duration("30m1h")
    def test_garbage(self):
        with self.assertRaises(ValueError): parse_duration("ten minutes")
'''

initial = {"spec": SPEC, "tests": TESTS, "code": "", "feedback": "", "attempts": 0, "passed": False}

# stream(..., stream_mode="updates") yields each node's output as it finishes: useful for showing progress.
final = dict(initial)
for step in code_graph.stream(initial, stream_mode="updates"):
    for node, update in step.items():
        final.update(update)
        if node == "generate":
            print(f"generate: attempt {update['attempts']}")
        else:
            print(f"test:     {'PASSED' if update['passed'] else 'failed'}")


In [ ]:
if final["passed"]:
    print(f"Passed after {final['attempts']} attempt(s):\n")
    print(final["code"])
else:
    print(f"No passing code after {final['attempts']} attempts. Last attempt:\n")
    print(final["code"])

**The last prompt sent to the model.** If the first attempt failed, it ends with the previous code and the test output: this is how the test node's result reaches the generate node on a retry.

In [ ]:
print(requests_sent[-1]["messages"][1]["content"])

## 7. Conversation memory with a checkpointer

A chat model is stateless: each call sees only the messages you send. In Week 6 you kept the history in a list yourself. With a checkpointer, the agent stores the state per `thread_id`, so each user or conversation gets its own history without extra code.

`InMemorySaver` is lost when the kernel restarts. For a deployed project, LangGraph provides SQLite and Postgres checkpointers with the same interface (packages `langgraph-checkpoint-sqlite` and `langgraph-checkpoint-postgres`).

In [ ]:
assistant = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant for a software project team. Be brief.",
    checkpointer=InMemorySaver(),
)

def chat(thread_id: str, text: str):
    out = assistant.invoke({"messages": [HumanMessage(text)]}, {"configurable": {"thread_id": thread_id}})
    return out["messages"][-1]

show(chat("team-a", "Our project is a code-review assistant for pull requests on GitHub."), label="team-a")
show(chat("team-a", "What is our project, in five words?"), label="team-a")
show(chat("team-b", "What is our project, in five words?"), label="team-b")   # different thread: no shared history

**Exercise.** Long conversations eventually exceed the context window. Add `SummarizationMiddleware` to the assistant (see the LangChain documentation for its parameters) and test it with a conversation of 20 or more turns.

## 8. Choosing the right level for your project

Use the simplest level that does the job. Each step down this list adds capability and adds code you must understand, test and debug.

| You need | Use | Section |
|---|---|---|
| One prompt in, text out | `llm.invoke`, or a prompt template chain | 1 |
| Data your code can act on | `with_structured_output` | 2 |
| Answers grounded in your documents | Retriever + chain | 3 |
| The model choosing among tools in a loop | `create_agent` | 4 |
| Guarantees around that loop (approval, limits, redaction) | Middleware | 5 |
| A workflow with its own fixed steps, branches or retry policy | A LangGraph `StateGraph` | 6 |
| State that persists across calls | A checkpointer | 5, 7 |

**What stays your responsibility, whichever level you choose:**
* **Evaluation.** None of these components tells you whether the output is correct. Build a test set early.
* **Tool safety.** Every tool is an action the model can take with untrusted arguments. Validate inputs, sandbox side effects, and gate anything destructive behind human approval.
* **Verification.** An agent's claim that it succeeded is not evidence. Check results with code (Section 5's final cell, Section 6's tests).
* **Traceability.** Keep traces of what the AI did (Section 4). You will need them for debugging and for your disclosure artefact.
* **Debugging through layers.** When something fails, print the actual messages sent to the model. Most "framework bugs" turn out to be prompts or tool descriptions that are not what you thought.